# Fixed shorter training, experiment 30

Train the same CLAP graph architecture for 20 epochs with cosine T_max=20.
Reuse the verified 60-epoch fold preprocessing exactly and compare with the
immutable experiment-29 selected control. No early stopping or checkpoint grid.
Five whole-window folds, 60-second purge, three seeds per fold and final fit.
Write the OOF spellF1 decision before scoring only its winner on reused main.
All data, labels and model outputs remain private; no competition submission.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_early');WORK.mkdir(parents=True,exist_ok=True)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
configs=list(INPUT.rglob('early_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
roots=[p.parent for p in INPUT.rglob('F1_TARGET.md') if (p.parent/'relational_possession/early.py').is_file()];assert len(roots)==1
CODE=roots[0]
for relative,digest in plan['source_sha256'].items():assert sha(CODE/relative)==digest
baseplans=list(INPUT.rglob('clap_input.json'));assert len(baseplans)==1;BASE=baseplans[0].parent
FEATURES=BASE/'features.npz';GRAPH=BASE/'graph.npz';OWNER=BASE/'owner_scores.npz'
AUDIO=next(INPUT.rglob('clap_audio.npz'))
for name,path in [('features',FEATURES),('graph',GRAPH),('owner_scores',OWNER),('audio',AUDIO)]:assert sha(path)==plan['fingerprints'][name]
sources=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='matched_frozen_contrastive_audio'];assert len(sources)==1
SOURCE=sources[0]
controls=[p.parent for p in INPUT.rglob('decision.json') if p.parent.name=='early_control'];assert len(controls)==1
CONTROL=controls[0]
for relative,digest in plan['control_sha256'].items():assert sha(CONTROL/relative)==digest
labels=[p.parent for p in INPUT.rglob('train.csv') if p.parent.name=='clap_labels'];assert len(labels)==1;LABELS=labels[0]
for name in ('train','validation'):assert sha(LABELS/f'{name}.csv')==plan['fingerprints'][name+'_labels']
DEPS=Path('/tmp/barca_early_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',HF_HUB_OFFLINE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full suite passed; starting fixed shorter training.',flush=True)


In [ ]:
RUN=WORK/'study'
run('-m','relational_possession.early','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,'--owner-scores',OWNER,
 '--source-study',SOURCE,'--control-study',CONTROL,'--train-labels',LABELS/'train.csv','--validation-labels',LABELS/'validation.csv',
 '--out',RUN,'--device','cuda','--bootstrap','1000')


In [ ]:
report=json.loads((RUN/'report.json').read_text());assert not report['experiment']['validation_used_for_selection']
protocol=dict(git_checkpoint=plan['git_checkpoint'],source_sha256=plan['source_sha256'],fingerprints=plan['fingerprints'],
 control_sha256=plan['control_sha256'],torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
 selected_csv_sha256=sha(RUN/'selected_predictions.csv'),fixed_epochs=20,reference_files_unchanged=all(sha(LABELS/f'{n}.csv')==plan['fingerprints'][n+'_labels'] for n in ('train','validation')))
assert protocol['reference_files_unchanged']
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('EARLY GPU RUN COMPLETE',flush=True)
